# Qwen-Image 2.1 Uncensored (GGUF) — ComfyUI on Colab

Model: [abenzerps/Qwen-Image-2.1-Uncensored-GGUF](https://huggingface.co/abenzerps/Qwen-Image-2.1-Uncensored-GGUF)

**Runtime → Change runtime type → GPU** (L4 recommended; T4 works with `lowvram=True`, A100 for Q8_0 + bf16 encoder).

Steps: get code → setup → download models → launch → open the ComfyUI link.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!free -h | head -2
!df -h / | tail -1

## 1. Get the code

Clones [AndreRese/NewGwen2.1](https://github.com/AndreRese/NewGwen2.1). Re-running the cell pulls the latest version.

In [ ]:
import os
REPO_URL = "https://github.com/AndreRese/NewGwen2.1"
PROJECT_DIR = "/content/qwen21"

if os.path.isdir(os.path.join(PROJECT_DIR, ".git")):
    !git -C {PROJECT_DIR} pull -q --ff-only
else:
    !git clone -q {REPO_URL} {PROJECT_DIR}
%cd {PROJECT_DIR}
!ls

## 2. ComfyUI + ComfyUI-GGUF (leejet fork) + dependencies

Uses ComfyUI **master** (needed for `TextEncodeQwenImage21`) and the **leejet** fork of ComfyUI-GGUF (the only one with Qwen-Image 2.1 support right now).

In [ ]:
!bash setup_colab.sh

## 3. Download models (~14 GB default)

Only the selected quant is downloaded.

| `QUANT` | size | | `TEXT_ENCODER` | size |
|---|---|---|---|---|
| Q4_0 | 4.05 GB | | int8 (recommended) | 9.35 GB |
| **Q4_K_M** | 4.60 GB | | bf16 | 17.5 GB |
| Q8_0 | 7.59 GB | | w4a8 (experimental) | 6.3 GB |

In [ ]:
QUANT = "Q4_K_M"        #@param ["Q4_0", "Q4_K_M", "Q8_0"]
TEXT_ENCODER = "int8"   #@param ["int8", "bf16", "w4a8"]

!python download_models.py --quant {QUANT} --text-encoder {TEXT_ENCODER}

## 4. Launch ComfyUI (+ Gradio)

- **`GRADIO = True`** (default): prints two links — the ComfyUI canvas via Colab's port proxy, and a public **`*.gradio.live`** URL with a simple prompt → image UI that drives ComfyUI through its API (same style as the Wan2.2 app). The Gradio page also shows the ComfyUI host status (GPU memory, queue) and the canvas link. This cell keeps running while the app is up — stop it to shut Gradio down.
- **`GRADIO = False`**: only ComfyUI, via the proxy link.

In the ComfyUI canvas: **Workflow → Open → `qwen_image_2.1_gguf_t2i`** (or drag `workflows/qwen_image_2.1_gguf_t2i.json` onto it). If your `QUANT` isn't `Q4_K_M`, pick your file in the *Unet Loader (GGUF)* node.

In [ ]:
LOWVRAM = False          #@param {type:"boolean"}   (turn on for T4 / 15 GB)
CLOUDFLARE_TUNNEL = False #@param {type:"boolean"}   (extra public *.trycloudflare.com URL)
GRADIO = True             #@param {type:"boolean"}   (also start the Gradio app with a public *.gradio.live link)
RESTART = False           #@param {type:"boolean"}   (kill a running ComfyUI first — after pulling updates / patches)

if GRADIO:
    # starts ComfyUI (if not running) + a Gradio front-end: prompt -> image via the ComfyUI API,
    # with a status panel showing the ComfyUI host URL / GPU / queue. Blocks this cell while it runs.
    from app_gradio import main
    main(lowvram=LOWVRAM, tunnel=CLOUDFLARE_TUNNEL, restart=RESTART)
else:
    from launch_comfyui import launch
    url = launch(lowvram=LOWVRAM, tunnel=CLOUDFLARE_TUNNEL, restart=RESTART)

## 5. (Optional) Headless generation from a cell

Only usable when the launch cell above is **not** blocking (i.e. `GRADIO = False`). Talks to the running server through the ComfyUI API using `workflows/qwen_image_2.1_gguf_t2i_api.json`.

In [ ]:
PROMPT = "Cinematic photo of a woman in a red dress on a rooftop at dusk, city bokeh, 85mm, film grain"  #@param {type:"string"}
NEGATIVE = ""      #@param {type:"string"}
WIDTH = 1024       #@param {type:"integer"}
HEIGHT = 1024      #@param {type:"integer"}
STEPS = 25         #@param {type:"integer"}
CFG = 1.0          #@param {type:"number"}
SEED = -1          #@param {type:"integer"}   (-1 = random)

from generate import run
from IPython.display import Image, display
files, used_seed = run(PROMPT, NEGATIVE, WIDTH, HEIGHT, STEPS, CFG, None if SEED < 0 else SEED,
            gguf=f"qwen-image-2.1-{QUANT}.gguf")
print('seed', used_seed)
for f in files:
    display(Image(f))

## Troubleshooting

```python
!tail -50 ComfyUI/comfyui.log                                  # server log
!git -C ComfyUI pull && git -C ComfyUI/custom_nodes/ComfyUI-GGUF pull   # missing nodes / unknown arch
```

- **`Expected weight to be of same shape as normalized_shape … [136] … [128]`** → the **Q8_0** file (maybe Q4_0 too) has quantized 1D norm weights ([HF discussion #4](https://huggingface.co/abenzerps/Qwen-Image-2.1-Uncensored-GGUF/discussions/4)). Either switch to **Q4_K_M**, or apply the loader patch and restart ComfyUI (already done by `setup_colab.sh` in fresh sessions):
  ```python
  !git -C /content/qwen21 pull -q && python patch_gguf_loader.py
  ```
  then stop the launch cell and re-run it with `RESTART = True`.
- **Red `TextEncodeQwenImage21` node** → ComfyUI too old, pull master.
- **GGUF "unknown architecture"** → ComfyUI-GGUF not the leejet fork or stale, pull.
- **OOM (T4)** → `LOWVRAM = True`, stay at 1024², try `TEXT_ENCODER = "w4a8"`.
- **Proxy link 403** → re-run the launch cell (it only reprints the link) or enable the cloudflare tunnel.
- Save outputs: `!zip -r outputs.zip ComfyUI/output` then download from the Files panel.